# nAChR VEP — TODO & Status

**Last updated:** 2026-07-14 (AlphaFold structures added)  
**Context:** Comparison of nAChR VEP (in progress) vs ENaC VEP (complete/done)

---

## Current Baseline (nAChR)

| Metric | Value |
|--------|-------|
| Mutations | 351 (218 LOF / 133 GOF) |
| Features | 52 (24 physicochemical + 3 substitution + 17 positional + 8 structural) |
| Subunits | 16 human nAChR subunits |
| PDB structures | 4 experimental + 6 AlphaFold = **10 total (100% subunit coverage)** |
| Structural coverage | **305/351 mapped (87%)**, 46 imputed |
| Models run | 4 (LR, SVM-RBF, RF, LightGBM) |
| Best F1 | 0.66 (Logistic Regression) |
| Species | Human only (mouse/rat being collected) |

---

## What's Already Done ✅

| # | Component | Notes |
|---|-----------|-------|
| 1 | Config system | 16 subunits, PDB_MAPPING, CANONICAL_ACCESSIONS, feature groups |
| 2 | Data loader | Excel→standardized columns, LOF/GOF binary, FASTA sequences |
| 3 | Physicochemical features | 8 AAIndex scales × {wt, mt, diff} = 24 features |
| 4 | Substitution features | BLOSUM62 + **Grantham distance** (nAChR-exclusive) |
| 5 | Structural features | 8 features: RSA, B-factor, DSSP×3, C-beta, **HSE×2** — multi-PDB, Shrake-Rupley SASA (no mkdssp) |
| 6 | Feature encoder | sklearn-compatible NachrFeatureEncoder |
| 7 | Model registry | 9 models with Optuna HP spaces |
| 8 | Nested CV | Multi-seed 5×5 nested CV + simple CV mode |
| 9 | Experiment runner | CLI with --quick, --model, --all-models, --no-structural |
| 10 | PDB files | 7QKO.cif, 7EKI.cif, 6CNJ.cif, 6PV7.cif |
| 11 | FASTA sequences | All 16 subunit folders with canonical RefSeq isoforms |
| 12 | Initial results | 4 models saved to results/ |

---

## TODO List

### 🔴 CRITICAL — Core experiments blocked without these

| # | Task | Status | What ENaC Has | Notes |
|---|------|--------|---------------|-------|
| 1 | **Mouse + Rat data integration** | 🟡 Collecting | Human+Mouse CSV with species column | nAChR only loads human Excel. Need mouse/rat DB mapped to human positions |
| 2 | **AlphaFold structures for 6 subunits** | ✅ DONE (2026-07-14) | 1 PDB covers all 3 ENaC subunits | All 6 downloaded (v6 CIFs), config updated, verified: 305/351 mapped (87%) |
| 3 | **Species feature in encoder** | ⬜ TODO | species_mouse/species_human one-hot | Needed once mouse/rat data is added |
| 4 | **Species Transfer CV** | ⬜ TODO | 3-condition test (human-only / mouse-only / mixed) | Core experiment; port from ENaC, adapt for binary |

### 🟡 HIGH — Needed for fair comparison & paper quality

| # | Task | Status | What ENaC Has | Notes |
|---|------|--------|---------------|-------|
| 5 | **Multiple encoding strategies** | ⬜ TODO | ordinal, onehot, fullseq, engineered, engineered_original, combined | nAChR only has engineered; need for data-driven vs domain-driven |
| 6 | **Evaluation module** (evaluation.py) | ⬜ TODO | compute_metrics, Wilcoxon/t-test, SHAP, beeswarm plots | nAChR just saves JSON — no stats/plots |
| 7 | **Feature caching** (precompute_feature_cache) | ⬜ TODO | Avoids re-extracting per model | Currently extracts from scratch each time |
| 8 | **Noah's original thesis features** | ⬜ TODO | 31 features: 10 AA props, MSA-aligned positions | Port aa_features.py + position_features.py |
| 9 | **CombinedFeatureEncoder** | ⬜ TODO | Deduplicated merge (37+31→50 features) | Enables feature-source ablation |

### 🟢 MEDIUM — Improves analysis depth

| # | Task | Status | Notes |
|---|------|--------|-------|
| 10 | **Ablation studies** (run_ablation.py) | ⬜ TODO | Leave-one-group-out; config already defines FEATURE_GROUPS |
| 11 | **Feature importance** (SHAP) | ⬜ TODO | SHAP TreeExplainer + permutation + bar/beeswarm plots |
| 12 | **Pseudo-resolution for terminal IDRs** | ⬜ TODO | N/C-terminal tails → coil + chain_max B-factor |

### 🔵 LOW — Nice to have / end-stage

| # | Task | Status | Notes |
|---|------|--------|-------|
| 13 | CatBoost support + subprocess isolation | ⬜ TODO | Prevents C++ segfaults on long runs |
| 14 | Config YAML support | ⬜ TODO | Config.from_yaml()/to_yaml() |
| 15 | Paper figure generation scripts | ⬜ TODO | Auto-generate figures + LaTeX tables from results |
| 16 | HPC/SLURM support | ⬜ TODO | resolve_outdir.py, SLURM scripts |

### ✅ N/A — nAChR already does this better

| # | Item | Why Skip |
|---|------|----------|
| — | DSSP binary dependency | nAChR uses Shrake-Rupley (pure Python) — no mkdssp needed |

---

## Critical Path (Recommended Order)

```
1. Mouse + Rat data collection & mapping    ← IN PROGRESS (you)
2. AlphaFold structures for 6 subunits      ← ✅ DONE (2026-07-14)
3. Species feature in encoder               ← NEXT
4. Species Transfer CV (port from ENaC)
5. Evaluation module (port evaluation.py)
6. Feature caching + multiple encodings
7. Ablation studies + feature importance
8. Paper figures & writeup
```

---

## Key Structural Differences (nAChR vs ENaC)

| Aspect | ENaC | nAChR |
|--------|------|-------|
| Subunits | 3 (α, β, γ) trimer | 16 (α1-10, β1-4, δ, ε, γ) pentamer |
| PDBs | 1 (6BQN) | **4 experimental + 6 AlphaFold = 10** (100% coverage) |
| Species | Human + Mouse | Human only (mouse/rat being collected) |
| Labels | 3-class (LOF/NnE/GOF) | Binary (LOF/GOF) |
| Structural features | 5 | 8 (+HSE up/down) |
| DSSP method | Requires mkdssp binary | Shrake-Rupley (no external dep) |
| Substitution features | 2 (BLOSUM62 only) | 3 (+Grantham distance) |
| Data source | CSV | Excel |

---

## AlphaFold Structure Download (Task #2) — COMPLETED

All 6 AlphaFold v6 CIFs downloaded to `data/raw/structure_files/`.
Download script: `scripts/download_alphafold_structures.py`

| Subunit | UniProt ID | CIF File | Chain | Residues | Status |
|---------|-----------|----------|-------|----------|--------|
| CHRNA2 | Q15822 | AF-Q15822.cif | A | 529 | ✅ |
| CHRNA5 | P30532 | AF-P30532.cif | A | 468 | ✅ |
| CHRNA6 | Q15825 | AF-Q15825.cif | A | 494 | ✅ |
| CHRNA9 | Q9UGM1 | AF-Q9UGM1.cif | A | 479 | ✅ |
| CHRNA10 | Q13002 | AF-Q13002.cif | A | 908 | ✅ |
| CHRNB3 | Q05901 | AF-Q05901.cif | A | 458 | ✅ |

**Result:** Structural coverage improved from 267→305 mapped mutations (76%→87%).
46 mutations still imputed (alignment gaps, disordered regions).

**Important caveats for AlphaFold features:**
- B-factor = pLDDT (confidence score), NOT thermal mobility
- RSA/C-beta/HSE reflect **monomer** context, not pentamer assembly
- Inter-subunit interface features will differ from experimental PDBs
- `source: "alphafold"` tag in PDB_MAPPING allows downstream filtering